# XD-Violence Riot (B4) Pull + Validation

Pulls the riot-labeled (B4) clips from XD-Violence via the public
HuggingFace mirror (`jherng/xd-violence`) directly into Colab / Drive —
no OneDrive, no China-hosted mirror, no local machine involved.

**Verified before this notebook was written (see `docs/findings.md`):**
- The HF mirror serves individual files over plain HTTPS, no auth needed.
- 4750 total video files across 5 train dirs + `test_videos`; 485 are
  riot (B4)-labeled (9.78 GiB total) — small enough for one Colab pass.
- Filenames contain a literal `#`, which must be percent-encoded as
  `%23` in HTTP requests or the server 404s (everything after `#` gets
  silently treated as a URL fragment and dropped).
- Some clips carry combined labels (e.g. `label_B4-G-0`) — the filter
  below checks each hyphenated code, not just a `label_B4` substring.

**What this notebook does, in order:** list every video directory on the
HF repo -> filter for riot clips -> build a manifest -> dry-run 5 test
clips and validate them -> pull the rest -> validate the full set
(size check + can-it-actually-open check) -> show a visual spot-check
grid so you can eyeball that these are really riot footage, not junk.

## Step 0 — Setup

In [ ]:
!pip install requests opencv-python-headless -q

In [ ]:
import csv
import os
import re
import random
import urllib.parse

import requests
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

FINAL_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_riot'
os.makedirs(FINAL_DIR, exist_ok=True)
print('Saving to:', FINAL_DIR)

## Step 1 — List the HF repo and build the riot-clip manifest

No video data is downloaded in this step — only file listings (name +
size) via the HF tree API. Cheap and fast.

In [ ]:
REPO = "jherng/xd-violence"
DIRS = {
    "data/video/1-1004": "train",
    "data/video/1005-2004": "train",
    "data/video/2005-2804": "train",
    "data/video/2805-3319": "train",
    "data/video/3320-3954": "train",
    "data/video/test_videos": "test",
}
RIOT_CODE = "B4"
BASE_DL = f"https://huggingface.co/datasets/{REPO}/resolve/main/"


def is_riot(path: str) -> bool:
    """True if B4 is one of the (possibly hyphenated, co-occurring)
    label codes embedded in the filename, e.g. '..._label_B4-0-0.mp4'
    or '..._label_B4-G-0.mp4'."""
    m = re.search(r'_label_([A-Za-z0-9\-]+?)(?:-\d+-\d+)?\.\w+$', path)
    if not m:
        return False
    return RIOT_CODE in m.group(1).split('-')


def list_dir(dirpath: str):
    url = f"https://huggingface.co/api/datasets/{REPO}/tree/main/{urllib.parse.quote(dirpath)}?limit=1000"
    out = []
    while url:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        out.extend(r.json())
        link = r.headers.get("Link", "")
        m = re.search(r'<([^>]+)>;\s*rel="next"', link)
        url = m.group(1) if m else None
    return out

In [ ]:
manifest_rows = []
for d, split in DIRS.items():
    print(f"Listing {d} ...")
    files = list_dir(d)
    n_riot = 0
    for f in files:
        if f.get("type") != "file" or not is_riot(f["path"]):
            continue
        path = f["path"]
        vid_id = path.rsplit("/", 1)[-1]
        download_url = BASE_DL + urllib.parse.quote(path).replace("%2F", "/")
        manifest_rows.append({
            "id": vid_id, "label": "riot", "split": split,
            "size_bytes": f.get("size", 0), "url": download_url,
        })
        n_riot += 1
    print(f"  {len(files)} files total, {n_riot} riot (B4)")

total_gib = sum(r["size_bytes"] for r in manifest_rows) / (1024 ** 3)
print(f"\nFound {len(manifest_rows)} riot clips, {total_gib:.2f} GiB total.")

In [ ]:
manifest_path = os.path.join(FINAL_DIR, "manifest_xdviolence_riot.csv")
with open(manifest_path, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["id", "label", "split", "size_bytes", "url"])
    w.writeheader()
    w.writerows(manifest_rows)
print(f"Manifest saved to {manifest_path}")

df = pd.DataFrame(manifest_rows)
summary = df.groupby("split").agg(clips=("id", "count"),
                                    gib=("size_bytes", lambda s: s.sum() / 1024**3))
summary

## Step 2 — Dry run: download + validate 5 test-split clips first

Before spending time on the full 9.78 GiB pull, confirm downloads
actually work and produce playable video files.

In [ ]:
def download_one(row, dest_dir):
    dest = os.path.join(dest_dir, row["id"])
    if os.path.exists(dest) and os.path.getsize(dest) == row["size_bytes"]:
        return True  # already downloaded at the right size -- resumable across sessions
    try:
        with requests.get(row["url"], stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    fh.write(chunk)
        return True
    except Exception as e:
        print(f"  FAILED {row['id']}: {e}")
        if os.path.exists(dest):
            os.remove(dest)
        return False


test_rows = [r for r in manifest_rows if r["split"] == "test"][:5]
print(f"Dry run: downloading {len(test_rows)} test-split riot clips...")
dry_ok = [r for r in test_rows if download_one(r, FINAL_DIR)]
print(f"Dry run: {len(dry_ok)}/{len(test_rows)} downloaded.")

In [ ]:
# Validate: file size matches manifest, and OpenCV can actually open
# and read frames from it (catches truncated/corrupt downloads that a
# size check alone would miss).
def validate_clip(row, dest_dir):
    path = os.path.join(dest_dir, row["id"])
    result = {"id": row["id"], "exists": os.path.exists(path)}
    if not result["exists"]:
        return result
    result["size_on_disk"] = os.path.getsize(path)
    result["size_matches"] = result["size_on_disk"] == row["size_bytes"]
    cap = cv2.VideoCapture(path)
    result["opens"] = cap.isOpened()
    if result["opens"]:
        result["frame_count"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        result["fps"] = cap.get(cv2.CAP_PROP_FPS)
        result["width"] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        result["height"] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        ok, _frame = cap.read()
        result["first_frame_readable"] = ok
    cap.release()
    return result


dry_validation = pd.DataFrame([validate_clip(r, FINAL_DIR) for r in dry_ok])
dry_validation

In [ ]:
assert dry_validation["exists"].all(), "Some dry-run files are missing on disk"
assert dry_validation["size_matches"].all(), "Some dry-run files have the wrong size -- truncated download"
assert dry_validation["opens"].all(), "Some dry-run files won't open in OpenCV -- corrupt download"
assert dry_validation["first_frame_readable"].all(), "Some dry-run files opened but have no readable frames"
print("Dry run fully validated -- safe to pull the rest.")

## Step 3 — Download the remaining riot clips (~9.7 GiB, resumable)

Writes straight to the Drive-mounted path, so a mid-run disconnect
doesn't lose progress -- re-running this cell skips anything already
downloaded at the correct size.

In [ ]:
remaining = [r for r in manifest_rows if r not in dry_ok]
print(f"Downloading {len(remaining)} remaining riot clips...")
ok_count = 0
failed = []
for i, row in enumerate(remaining, 1):
    if download_one(row, FINAL_DIR):
        ok_count += 1
    else:
        failed.append(row["id"])
    if i % 25 == 0 or i == len(remaining):
        print(f"  {i}/{len(remaining)} processed ({ok_count} ok, {len(failed)} failed)")

print(f"\nDone. {ok_count}/{len(remaining)} downloaded.")
if failed:
    print("Failed IDs (re-run this cell to retry -- already-downloaded files are skipped):")
    for fid in failed:
        print(" ", fid)

## Step 4 — Validate the full pull

Two checks: every manifest row exists on disk at the right size, and
every file actually opens as a readable video (catches silent
corruption that a size-only check would miss).

In [ ]:
full_validation = pd.DataFrame([validate_clip(r, FINAL_DIR) for r in manifest_rows])

n = len(full_validation)
print(f"{n} clips in manifest")
print(f"  exists on disk:      {full_validation['exists'].sum()}/{n}")
print(f"  size matches:        {full_validation['size_matches'].sum()}/{n}")
print(f"  opens in OpenCV:     {full_validation['opens'].sum()}/{n}")
print(f"  first frame readable:{full_validation['first_frame_readable'].sum()}/{n}")

bad = full_validation[~(full_validation.get('exists', False)
                        & full_validation.get('size_matches', False)
                        & full_validation.get('opens', False)
                        & full_validation.get('first_frame_readable', False))]
if len(bad):
    print(f"\n{len(bad)} clip(s) need attention:")
    display(bad)
else:
    print("\nAll riot clips downloaded and validated clean.")

## Step 5 — Visual spot-check

Pulls the middle frame from a random sample of downloaded clips so you
can eyeball that this is actually riot/crowd-violence footage, not
black frames or something mislabeled.

In [ ]:
downloaded_ids = [r["id"] for r in manifest_rows
                  if os.path.exists(os.path.join(FINAL_DIR, r["id"]))]
sample_ids = random.sample(downloaded_ids, k=min(6, len(downloaded_ids)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, vid_id in zip(axes.flat, sample_ids):
    path = os.path.join(FINAL_DIR, vid_id)
    cap = cv2.VideoCapture(path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
    ok, frame = cap.read()
    cap.release()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(vid_id, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Next steps

- This manifest (`manifest_xdviolence_riot.csv` in `FINAL_DIR`) follows
  the same `id,label,split,...` shape as this project's other
  `data/manifest_*.csv` files — copy it into `data/` and adapt
  `scripts/build_*_manifest.py` conventions if this becomes the
  primary riot-positive source (see `docs/context.md`).
- Once validated, this real XD-Violence riot subset should replace the
  Capitol-riot stand-in for the *positive* class in a fresh
  `findings.md` entry — the RLVS-NonViolence negative-class stand-in
  question is separate and unaffected by this pull.
- No negative (Normal/A-labeled) class has been pulled here yet — this
  notebook only targets B4/riot on purpose, per the current scope.